# Airfoil Selection & Wing Aerodynamics
### Electric V-BAT Tail-Sitter -- Design Phase 2

---

**Purpose.** Select the wing airfoil section and verify that it satisfies
the aerodynamic requirements established by the conceptual sizing study.

**Upstream dependency.** This notebook reads `out/sizing_result.yaml`,
which is written by the final cell of `vbat_conceptual_design.ipynb`.
Run that notebook first if the file does not exist.

**What this notebook does NOT do.**
It does not re-run the mass closure loop. All sizing numbers are fixed.
The only decision made here is which NACA 4-digit section to use.

---

## Notebook Structure

| Section | What it does |
|---|---|
| 0 . Setup | Imports, paths, load sizing result |
| 1 . Design Point | The aerodynamic requirements the airfoil must satisfy |
| 2 . Candidate Comparison | Side-by-side table of NACA 4-digit candidates |
| 3 . Selected Airfoil | Full analysis of the chosen section |
| 4 . Constraint Check | Interpret warnings and design trades |
| 5 . Section Shape | Plot the airfoil geometry |
| 6 . Outputs | Write out/airfoil.yaml and coordinate .dat file |
| 7 . Impact on Sizing | How the selected L/D and CL_max compare to sizing assumptions |

---
## 0 . Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml

# -- Repo layout -----------------------------------------------------------
REPO_ROOT   = Path().resolve().parents[0]
SRC_PATH    = REPO_ROOT / "src"
CONFIG_PATH = REPO_ROOT / "config"
OUT_PATH    = REPO_ROOT / "out"
sys.path.insert(0, str(SRC_PATH))

# -- Module imports --------------------------------------------------------
from conceptual_design.airfoil_selection import (
    parse_naca4,
    analyse_airfoil,
    compare_airfoils,
    write_outputs,
    naca4_coordinates,
    AirfoilResult,
)

# -- Plot style ------------------------------------------------------------
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})
COLORS = ['#2c7bb6', '#d7191c', '#1a9641', '#f68b33', '#762a83']

# -- Check upstream file exists -------------------------------------------
sizing_file = OUT_PATH / "sizing_result.yaml"
if not sizing_file.exists():
    raise FileNotFoundError(
        f"out/sizing_result.yaml not found.\n"
        f"Run all cells in vbat_conceptual_design.ipynb first."
    )

print(f"Repo root   : {REPO_ROOT}")
print(f"Config path : {CONFIG_PATH}")
print(f"Out path    : {OUT_PATH}")
print("Imports     : OK")

---
## 1 . Design Point

Load the settled sizing numbers and state the requirements the airfoil
must satisfy.  These numbers are fixed -- the airfoil choice cannot change
them.  If the chosen section cannot meet the stall requirement the correct
response is to revisit `config/aerodynamics.yaml` (increase CL_max or
V_stall) and re-run the sizing notebook, not to fudge the numbers here.

### Requirements the airfoil must satisfy

| Requirement | Symbol | Source |
|---|---|---|
| Wing loading at design point | W/S | Size Matching Diagram |
| Stall speed | V_stall | config/aerodynamics.yaml |
| Cruise speed | V_cruise | config/mission.yaml |
| Aspect ratio | AR | config/aerodynamics.yaml |
| Minimum 3D CL_max | (W/S) / (0.5*rho*V_stall^2) | derived |
| Minimum t/c | 0.09 | spar depth constraint |

In [ ]:
# -- Load sizing result ----------------------------------------------------
with open(sizing_file) as f:
    sr = yaml.safe_load(f)

# -- Load airfoil selection config -----------------------------------------
with open(CONFIG_PATH / "airfoil_selection.yaml") as f:
    af_cfg = yaml.safe_load(f)

# -- Extract design-point numbers ------------------------------------------
WS_design    = sr["WS_design_N_m2"]     # N/m^2
V_cruise     = sr["V_cruise_ms"]         # m/s
V_stall      = sr["V_stall_ms"]          # m/s
AR           = sr["AR"]                  # -
rho          = sr["rho_kg_m3"]           # kg/m^3
MTOW_kg      = sr["MTOW_kg"]             # kg
b_wing       = sr["b_wing_m"]            # m
chord        = sr["chord_mean_m"]        # m
CD0_fuselage = af_cfg["CD0_fuselage"]    # -

# -- Derived stall requirement ---------------------------------------------
q_stall          = 0.5 * rho * V_stall**2
CL_max_required  = WS_design / q_stall

print("Design point loaded from out/sizing_result.yaml")
print()
print("Aerodynamic requirements for airfoil selection:")
print(f"  Wing loading (W/S)*    : {WS_design:.1f} N/m^2")
print(f"  Aspect ratio (AR)      : {AR:.1f}")
print(f"  Wing span              : {b_wing:.3f} m")
print(f"  Mean chord             : {chord:.4f} m")
print(f"  Cruise speed           : {V_cruise:.1f} m/s")
print(f"  Required stall speed   : {V_stall:.1f} m/s")
print(f"  Min CL_max (3D)        : {CL_max_required:.4f}")
print(f"  Min t/c (spar depth)   : 0.09")
print(f"  CD0 fuselage (assumed) : {CD0_fuselage:.4f}")

---
## 2 . Candidate Comparison

Four NACA 4-digit sections cover the relevant design space for a small
electric UAV at Re ~ 3e5..8e5:

| Designation | Camber | Notes |
|---|---|---|
| NACA 0012 | 0% | Symmetric. Zero pitching moment. Lowest CL_max. |
| NACA 2412 | 2% | Mild camber. Small pitching moment. Classic GA section. |
| NACA 4412 | 4% | Higher camber. Best CL_max at t/c=0.12. Larger Cm. |
| NACA 2415 | 2% | Thicker spar. Slightly more profile drag than 2412. |

All are evaluated at the design wing loading, cruise speed, and AR from
the sizing result.  The `Warns` column counts failed design constraints.

In [ ]:
# -- Run comparison --------------------------------------------------------
candidates = ["NACA 0012", "NACA 2412", "NACA 4412", "NACA 2415"]

results_all = [
    analyse_airfoil(d, AR, WS_design, V_cruise, V_stall,
                    rho=rho, CD0_fuselage=CD0_fuselage)
    for d in candidates
]

print(f"{'Designation':<14} {'t/c':>6} {'Cl_max2D':>10} {'CL_max3D':>10} "
      f"{'Cd0':>8} {'L/D':>8} {'Warns':>6}")
print("-" * 64)
for r in results_all:
    flag = "OK" if not r.warnings else f"{len(r.warnings)}!"
    print(f"{r.designation:<14} {r.t:>6.3f} {r.Cl_max_2D:>10.4f} "
          f"{r.CL_max_3D:>10.4f} {r.Cd0_section:>8.5f} "
          f"{r.LD_cruise:>8.2f} {flag:>6}")

print()
print(f"Min CL_max (3D) required : {CL_max_required:.4f}")
print(f"Sections meeting stall   : ",
      ", ".join(r.designation for r in results_all
                if r.CL_max_3D >= CL_max_required))

In [ ]:
# -- Visualise the comparison ----------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

labels   = [r.designation for r in results_all]
x        = np.arange(len(labels))
width    = 0.55

# Panel 1: CL_max
ax = axes[0]
bars = ax.bar(x, [r.CL_max_3D for r in results_all],
              width, color=COLORS[:4], edgecolor='white')
ax.axhline(CL_max_required, color='crimson', lw=1.5, ls='--',
           label=f'Required {CL_max_required:.3f}')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('CL_max (3D)  [-]')
ax.set_title('Max Lift Coefficient', fontweight='bold')
ax.legend(fontsize=8)
for bar, r in zip(bars, results_all):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{r.CL_max_3D:.3f}', ha='center', va='bottom', fontsize=8)

# Panel 2: Cruise L/D
ax = axes[1]
bars = ax.bar(x, [r.LD_cruise for r in results_all],
              width, color=COLORS[:4], edgecolor='white')
ax.axhline(sr["CL_max_assumed"] / 0.025, color='gray', lw=1.0, ls=':',
           label='Sizing assumption L/D')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Cruise L/D  [-]')
ax.set_title('Cruise Lift-to-Drag Ratio', fontweight='bold')
for bar, r in zip(bars, results_all):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{r.LD_cruise:.1f}', ha='center', va='bottom', fontsize=8)

# Panel 3: Profile drag Cd0
ax = axes[2]
bars = ax.bar(x, [r.Cd0_section for r in results_all],
              width, color=COLORS[:4], edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Section Cd_0  [-]')
ax.set_title('Profile Drag Coefficient', fontweight='bold')
for bar, r in zip(bars, results_all):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f'{r.Cd0_section:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Airfoil Candidate Comparison  --  AR={:.0f},  W/S={:.0f} N/m^2,  V_cruise={:.0f} m/s'.format(
    AR, WS_design, V_cruise), fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_PATH / 'airfoil_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 . Selected Airfoil -- Full Analysis

The selected section is taken from `config/airfoil_selection.yaml`.
To change the selection: edit that file and re-run this cell onwards.
Do not change the designation here in the notebook.

In [ ]:
# -- Load selected designation from config ---------------------------------
designation  = af_cfg["designation"]
n_coords     = af_cfg["n_coords"]

print(f"Selected airfoil : {designation}")
print(f"  (from config/airfoil_selection.yaml)")
print()

# -- Full analysis ---------------------------------------------------------
result = analyse_airfoil(
    designation  = designation,
    AR           = AR,
    WS_N_m2      = WS_design,
    V_cruise     = V_cruise,
    V_stall      = V_stall,
    rho          = rho,
    CD0_fuselage = CD0_fuselage,
)

result.print_summary()

---
## 4 . Constraint Check

Three constraints are evaluated automatically:

1. **Stall speed** -- 3D CL_max must allow W/S at V_stall.
   If this fails the section has insufficient lift; choose a more
   cambered section or accept a higher stall speed in the config.

2. **Spar depth** -- t/c >= 0.09 for a structurally viable main spar.
   Sections thinner than this are only used for tail surfaces.

3. **Section headroom** -- 2D Cl_max should have >= 5% margin above
   the 3D CL_max divided back to section level (inverse of Raymer 3-8).
   If this fails the finite-wing correction is leaving no safety margin.

In [ ]:
# -- Constraint summary ----------------------------------------------------
print("Constraint check for", result.designation)
print()

# 1. Stall
WS_stall_max = 0.5 * rho * V_stall**2 * result.CL_max_3D
stall_ok     = WS_design <= WS_stall_max
print(f"  Stall constraint:")
print(f"    CL_max (3D)            : {result.CL_max_3D:.4f}")
print(f"    Max W/S at V_stall     : {WS_stall_max:.1f} N/m^2")
print(f"    Design W/S             : {WS_design:.1f} N/m^2")
print(f"    Status                 : {'PASS' if stall_ok else 'FAIL -- choose higher-lift section'}")
print()

# 2. Spar depth
spar_ok = result.t >= 0.09
print(f"  Spar depth constraint:")
print(f"    t/c                    : {result.t:.3f}")
print(f"    Minimum (structural)   : 0.090")
print(f"    Status                 : {'PASS' if spar_ok else 'FAIL -- section too thin for main spar'}")
print()

# 3. Headroom
Cl_max_min  = result.CL_max_3D / 0.9
margin_ok   = result.Cl_max_2D >= Cl_max_min * 1.05
margin_pct  = (result.Cl_max_2D / Cl_max_min - 1.0) * 100
print(f"  Section headroom:")
print(f"    Section Cl_max (2D)    : {result.Cl_max_2D:.4f}")
print(f"    Required (inverse 3-8) : {Cl_max_min:.4f}")
print(f"    Margin                 : {margin_pct:+.1f}%")
print(f"    Status                 : {'PASS' if margin_ok else 'FAIL -- insufficient headroom'}")
print()

# -- Overall --
all_ok = stall_ok and spar_ok and margin_ok
print("=" * 45)
if all_ok:
    print(f"  {result.designation} satisfies all constraints.")
else:
    print(f"  WARNING: {result.designation} has failed constraints.")
    print(f"  Review the items marked FAIL above.")
print("=" * 45)

---
## 5 . Section Shape

Plot the airfoil profile at actual wing chord scale, plus the camber
line and thickness distribution.

In [ ]:
from conceptual_design.airfoil_selection import naca4_camber, naca4_thickness
import math

M, P, t = result.M, result.P, result.t

# -- Coordinates at actual wing chord -------------------------------------
xu, yu, xl, yl = naca4_coordinates(M, P, t, n=300)
c = chord   # mean aerodynamic chord [m]

# -- Camber line and thickness envelope -----------------------------------
xs_c = np.linspace(0, 1, 300)
yc   = [naca4_camber(x, M, P)[0] for x in xs_c]
yt   = [naca4_thickness(x, t)    for x in xs_c]

fig, axes = plt.subplots(2, 1, figsize=(11, 6),
                          gridspec_kw={'height_ratios': [3, 1]})

# -- Top panel: airfoil profile -------------------------------------------
ax = axes[0]
ax.plot([x*c for x in xu], [y*c for y in yu],
        color=COLORS[0], lw=2.0, label='Upper surface')
ax.plot([x*c for x in xl], [y*c for y in yl],
        color=COLORS[0], lw=2.0, linestyle='--', label='Lower surface')
ax.plot([x*c for x in xs_c], [y*c for y in yc],
        color=COLORS[1], lw=1.2, ls=':', label='Camber line')

# Annotate t/c and chord
ax.annotate('', xy=(c, 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='<->', color='gray', lw=1.0))
ax.text(c/2, -0.012*c, f'chord = {c:.4f} m', ha='center', fontsize=8, color='gray')

# Max thickness marker
yt_max_idx = int(0.3 * len(xs_c))   # max thickness near 30% chord for NACA 4-digit
ax.annotate('', xy=(xs_c[yt_max_idx]*c,  yt[yt_max_idx]*c),
                xytext=(xs_c[yt_max_idx]*c, -yt[yt_max_idx]*c),
            arrowprops=dict(arrowstyle='<->', color=COLORS[3], lw=1.0))
ax.text(xs_c[yt_max_idx]*c + 0.003, 0,
        f't/c = {t:.2f}', fontsize=8, color=COLORS[3], va='center')

ax.set_aspect('equal')
ax.set_xlabel('x  [m]', fontsize=10)
ax.set_ylabel('y  [m]', fontsize=10)
ax.set_title(f'{result.designation}  --  chord = {c:.4f} m,  span = {b_wing:.3f} m',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')

# -- Bottom panel: thickness distribution ---------------------------------
ax2 = axes[1]
ax2.plot([x*c for x in xs_c], [y*c for y in yt],
         color=COLORS[2], lw=1.8, label='Half-thickness y_t')
ax2.fill_between([x*c for x in xs_c], 0, [y*c for y in yt],
                 alpha=0.15, color=COLORS[2])
ax2.set_xlabel('x  [m]', fontsize=10)
ax2.set_ylabel('y_t  [m]', fontsize=10)
ax2.set_title('Thickness distribution', fontsize=10)
ax2.set_xlim(0, c)

plt.tight_layout()
plt.savefig(OUT_PATH / 'airfoil_profile.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Profile area (unit chord) ~ {0.6826 * t:.5f}  (approx, symmetric)")
print(f"Max thickness             : {t*c*1000:.2f} mm  at chord = {c*1000:.1f} mm")
print(f"Max thickness / spar min  : {t/0.09:.2f}  (>1.0 = spar depth OK)")

---
## 6 . Outputs

Write the two output files to `out/`:

- `out/airfoil.yaml`             -- aerodynamic properties consumed by `wing_sizing`
- `out/airfoils/<name>.dat`      -- Selig-format coordinates for `cad/wing_profile.py`

These files are auto-generated.  Do not edit them by hand.
To regenerate: change `config/airfoil_selection.yaml` and re-run from Section 3.

In [ ]:
# -- Write outputs to out/ ------------------------------------------------
write_outputs(
    result    = result,
    out_dir   = OUT_PATH,
    n_coords  = n_coords,
    chord     = chord,   # scale to actual wing chord
)

print()
print("out/airfoil.yaml contents:")
print("-" * 40)
print(open(OUT_PATH / "airfoil.yaml").read())

---
## 7 . Impact on Sizing Assumptions

The conceptual sizing notebook used assumed values for `L/D` and `CL_max`
in `config/aerodynamics.yaml`.  Now that the airfoil is selected those
numbers have physics-backed values.

This section compares the two and quantifies the impact.  If the
discrepancy is significant (>10%) the sizing assumptions should be updated
and the sizing loop re-run.

In [ ]:
# -- Load original sizing assumptions -------------------------------------
LD_assumed    = 8.0   # from config/aerodynamics.yaml
CL_max_assumed = sr["CL_max_assumed"]

# -- Computed values from airfoil analysis --------------------------------
LD_computed   = result.LD_cruise
CL_max_computed = result.CL_max_3D

delta_LD      = (LD_computed   - LD_assumed)    / LD_assumed    * 100
delta_CLmax   = (CL_max_computed - CL_max_assumed) / CL_max_assumed * 100

print("Comparison: sizing assumptions vs physics-backed values")
print()
print(f"  {'Parameter':<22} {'Assumed':>10} {'Computed':>10} {'Delta':>8}")
print("-" * 54)
print(f"  {'L/D (cruise)':<22} {LD_assumed:>10.2f} {LD_computed:>10.2f} {delta_LD:>+7.1f}%")
print(f"  {'CL_max (3D)':<22} {CL_max_assumed:>10.4f} {CL_max_computed:>10.4f} {delta_CLmax:>+7.1f}%")
print()

# -- Guidance --------------------------------------------------------------
if abs(delta_LD) > 10:
    print(f"  WARNING: L/D differs by {delta_LD:+.1f}%.")
    print(f"  Update 'LD: {LD_computed:.1f}' in config/aerodynamics.yaml")
    print(f"  and re-run vbat_conceptual_design.ipynb to close the loop.")
else:
    print(f"  L/D assumption is within 10% -- sizing result remains valid.")

if abs(delta_CLmax) > 10:
    print(f"  WARNING: CL_max differs by {delta_CLmax:+.1f}%.")
    print(f"  Update 'CL_max: {CL_max_computed:.2f}' in config/aerodynamics.yaml")
    print(f"  and re-run vbat_conceptual_design.ipynb to close the loop.")
else:
    print(f"  CL_max assumption is within 10% -- sizing result remains valid.")

print()

# -- Rough MTOW sensitivity ------------------------------------------------
# Battery energy is proportional to 1/LD for cruise power;
# a 10% L/D improvement -> ~5% MTOW reduction (from sensitivity analysis)
dMTOW_LD = -delta_LD * 0.5   # rough elasticity from sensitivity notebook
print(f"  Estimated MTOW impact from L/D change  : {dMTOW_LD:+.1f}%")
print(f"  Estimated MTOW impact from CL_max change: negligible (stall speed not binding)")
print(f"  Current MTOW (from sizing)             : {MTOW_kg:.3f} kg")
print(f"  Adjusted MTOW estimate                 : {MTOW_kg * (1 + dMTOW_LD/100):.3f} kg")

---
## Conclusions

### What this analysis established

1. **Stall constraint drives the section choice.**  With a design wing
   loading of ~123 N/m^2 and a 12 m/s stall speed, the minimum 3D
   CL_max required is ~1.40.  Of the candidates compared, only
   NACA 4412 comfortably clears this threshold.  NACA 2412 falls
   marginally short; NACA 0012 is well below.

2. **Camber trades lift for pitching moment.**  Moving from 2% to 4%
   camber (2412 -> 4412) adds 0.33 to CL_max (3D) at no cost in
   profile drag.  The penalty is a larger nose-down Cm_0 which the
   tail must trim, adding a small induced drag increment not captured
   here.  For a preliminary study the trim drag is second order.

3. **t/c = 0.12 is a comfortable structural baseline.**  All four
   candidates clear the 0.09 minimum.  At 0.12 the maximum spar
   thickness is ~20 mm for the chord sizes computed here, which
   accepts a standard 16 mm CFRP tube spar.

4. **L/D from physics is close to the sizing assumption.**  The
   parabolic polar at the design CL gives L/D ~ 13.  The sizing
   notebook assumed L/D = 8 -- a conservative number that implicitly
   absorbed interference drag and non-ideal cruise.  The discrepancy
   means the sizing is conservative; the true MTOW is likely slightly
   lower.  This is the correct direction for a first-principles study.

### Next steps

| Step | Action |
|---|---|
| Update L/D in config | Change `LD` in `config/aerodynamics.yaml` to computed value and re-run sizing |
| Wing detailed design | Use `out/airfoil.yaml` + `out/airfoils/<name>.dat` as inputs to wing structural sizing |
| Reynolds number check | Confirm Re at cruise is in the validity range (Re ~ chord * V / 1.5e-5) |
| XFOIL / panel code | Validate section polars at operating Re for higher fidelity |
| Tail surface selection | Symmetric section (NACA 0006 or 0009) for H-tail surfaces |

---
*This notebook uses only first-principles models from `src/conceptual_design/`.*
*All design choices are made in `config/airfoil_selection.yaml`.*
*Outputs are written to `out/` and should not be edited by hand.*